# Kubernetes Replay Analysis Pipeline

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    import seaborn as sns  # type: ignore
except ModuleNotFoundError:  # pragma: no cover - fallback for lean envs
    sns = None

# Ensure the repository root is importable so ``common`` resolves reliably.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "common").exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / "common").exists():
            REPO_ROOT = candidate
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.analysis import (
    bootstrap_ci,
    load_and_harmonise,
    strongest_baseline,
    summarise,
    write_run_metadata,
)

if sns is not None:
    sns.set_theme(style="whitegrid")
else:
    plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
BACKEND = "k8s"
ASSETS_DIR = REPO_ROOT / "assets"
RESULT_PATTERNS = [
    'analysis/k8s_results/**/*.csv',
    'analysis/k8s_results/**/*.json',
    'kubenergysched/results_k8s/**/*.csv',
    'kubenergysched/results_k8s/**/*.json',
    'kubenergysched/results_k8s_tuned/**/*.csv',
    'kubenergysched/results_k8s_tuned/**/*.json',
    'k8s/results/**/*.csv',
    'k8s/results/**/*.json'
]
MANIFEST_PATH = ASSETS_DIR / "sweep_manifest.json"

ASSETS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_policy_colors(policies):
    base = {
        "k8s-default": "#1f77b4",
        "carbonscaler": "#ff7f0e",
        "keids": "#2ca02c",
        "topsis": "#9467bd",
        "hetpolicy": "#8c564b",
        "themis-base": "#17becf",
    }
    if sns is not None:
        palette = sns.color_palette("Set2", n_colors=len(policies) or 1)
    else:
        cmap = plt.get_cmap("tab10")
        palette = [cmap(idx % cmap.N) for idx in range(len(policies) or 1)]
    colors = {}
    for idx, policy in enumerate(policies):
        colors[policy] = base.get(policy, palette[idx % len(palette)])
    return colors


def pareto_front(points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=float)
    mask = np.ones(len(points), dtype=bool)
    for i, point in enumerate(points):
        if not mask[i]:
            continue
        dominated = (
            (points[:, 0] <= point[0])
            & (points[:, 1] <= point[1])
            & ((points[:, 0] < point[0]) | (points[:, 1] < point[1]))
        )
        mask &= ~dominated
    return mask


def apply_manifest_filter(frame: pd.DataFrame, backend: str, manifest_path: Path) -> pd.DataFrame:
    manifest = {}
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
    policy_rows = frame[frame["site"].isna()].copy()
    keys = sorted({key for key in policy_rows["sweep_key"].dropna()})
    manifest[backend] = keys
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    other = "k8s" if backend == "sim" else "sim"
    if other in manifest and manifest[other]:
        shared = set(keys) & set(manifest[other])
        if shared:
            frame = frame[frame["sweep_key"].isin(shared) | frame["sweep_key"].isna()].copy()
    return frame

In [ ]:
df = load_and_harmonise(RESULT_PATTERNS)
if df.empty:
    raise RuntimeError("No runs matched the glob patterns; adjust RESULT_PATTERNS.")

df["backend"] = BACKEND

numeric_columns = [
    "jobs",
    "energy_kwh",
    "carbon_gco2e",
    "makespan_min",
    "latency_p50_s",
    "latency_p95_s",
    "assigned_jobs",
]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["jobs"] = df["jobs"].astype(float)
df["energy_per_job_kwh"] = df["energy_kwh"] / df["jobs"]
df["carbon_per_job_g"] = df["carbon_gco2e"] / df["jobs"]
df["jobs_per_kwh"] = df["jobs"] / df["energy_kwh"]

df["sweep_key"] = df.apply(
    lambda row: (
        f"{row['policy']}::" + str(int(round(row["jobs"])))
        if pd.notna(row["jobs"])
        else None
    ),
    axis=1,
)

df = apply_manifest_filter(df, BACKEND, MANIFEST_PATH)
df.drop(columns=["sweep_key"], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

In [ ]:
summary = summarise(df.copy(), assets_dir=ASSETS_DIR)
summary_path = ASSETS_DIR / f"{BACKEND}_summary_table.csv"
summary.to_csv(summary_path, index=False)

policy_order = [col for col in summary.columns if col != "metric"]
policy_colors = get_policy_colors(policy_order)
baseline_policy = strongest_baseline(df, BACKEND)
effect_path = ASSETS_DIR / f"{BACKEND}_effect_sizes.csv"

summary_path, effect_path, baseline_policy

In [ ]:
policy_rows = df[df["site"].isna()].copy()
site_rows = df[df["site"].notna() & df["assigned_jobs"].notna()].copy()

pareto_data = (
    policy_rows.dropna(subset=["energy_kwh", "carbon_gco2e"])
    .groupby("policy")[['energy_kwh', 'carbon_gco2e']]
    .median()
)
latency_data = policy_rows.dropna(subset=["latency_p95_s"])
makespan_data = {
    policy: policy_rows.loc[policy_rows["policy"] == policy, "makespan_min"].dropna()
    for policy in policy_order
}

pareto_data

In [ ]:
pareto_path = ASSETS_DIR / f"{BACKEND}_pareto_energy_vs_sci.png"
if pareto_data.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.text(0.5, 0.5, "Energy/carbon data unavailable", ha="center", va="center")
    ax.axis("off")
else:
    ordered = pareto_data.reindex(policy_order).dropna()
    points = ordered.to_numpy()
    mask = pareto_front(points)
    fig, ax = plt.subplots(figsize=(7, 5))
    legend_labels = set()
    for (policy, row), is_front in zip(ordered.iterrows(), mask):
        energy = row["energy_kwh"]
        carbon = row["carbon_gco2e"]
        label = policy if policy not in legend_labels else None
        ax.scatter(
            energy,
            carbon,
            s=90 if is_front else 60,
            c=policy_colors.get(policy, "#333333"),
            alpha=0.85,
            edgecolor="black" if is_front else "none",
            linewidth=1.0 if is_front else 0.0,
            label=label,
        )
        legend_labels.add(policy)
        ax.annotate(policy, (energy, carbon), textcoords="offset points", xytext=(6, 4))
    front = ordered[mask]
    if len(front) > 1:
        hull = front.sort_values(["energy_kwh", "carbon_gco2e"])
        ax.plot(hull["energy_kwh"], hull["carbon_gco2e"], color="#444444", linestyle="--", linewidth=1)
    ax.set_xlabel("Energy (kWh)")
    ax.set_ylabel("Carbon (gCO₂e)")
    ax.set_title("Pareto frontier: energy vs. carbon")
    ax.grid(True, axis="both", alpha=0.3)
    if legend_labels:
        ax.legend(title="Policy", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(pareto_path, dpi=300, bbox_inches="tight")
plt.close(fig)
pareto_path

In [ ]:
tail_path = ASSETS_DIR / f"{BACKEND}_tail_latency_violin.png"
fig, ax = plt.subplots(figsize=(8, 5))
if latency_data.empty:
    ax.text(0.5, 0.5, "Latency data unavailable", ha="center", va="center")
    ax.axis("off")
elif sns is None:
    distributions = [
        latency_data.loc[latency_data["policy"] == policy, "latency_p95_s"].dropna().to_numpy()
        for policy in policy_order
    ]
    boxplots = ax.boxplot(
        distributions,
        labels=policy_order,
        patch_artist=True,
    )
    for patch, policy in zip(boxplots['boxes'], policy_order):
        patch.set_facecolor(policy_colors.get(policy, "#cccccc"))
    ax.set_ylabel("Latency p95 (s)")
    ax.set_title("Tail latency distribution per policy")
else:
    sns.violinplot(
        data=latency_data,
        x="policy",
        y="latency_p95_s",
        order=policy_order,
        palette=policy_colors,
        cut=0,
        inner="quart",
        ax=ax,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Latency p95 (s)")
    ax.set_title("Tail latency distribution per policy")
ax.set_xlabel("")
fig.tight_layout()
fig.savefig(tail_path, dpi=300, bbox_inches="tight")
plt.close(fig)
tail_path

In [ ]:
makespan_path = ASSETS_DIR / f"{BACKEND}_makespan_bars.png"
stats = []
for policy in policy_order:
    series = makespan_data.get(policy, pd.Series(dtype=float))
    if series.empty:
        stats.append((policy, np.nan, (np.nan, np.nan)))
    else:
        ci_low, ci_high = bootstrap_ci(series, np.median)
        stats.append((policy, float(series.median()), (ci_low, ci_high)))
stats_df = pd.DataFrame(stats, columns=["policy", "median", "ci"])

fig, ax = plt.subplots(figsize=(8, 5))
if stats_df["median"].isna().all():
    ax.text(0.5, 0.5, "Makespan data unavailable", ha="center", va="center")
    ax.axis("off")
else:
    centers = np.arange(len(stats_df))
    heights = stats_df["median"].to_numpy()
    lower = stats_df["ci"].apply(lambda c: c[0]).to_numpy()
    upper = stats_df["ci"].apply(lambda c: c[1]).to_numpy()
    yerr = np.vstack([heights - lower, upper - heights])
    ax.bar(
        centers,
        heights,
        color=[policy_colors.get(p, "#444444") for p in stats_df["policy"]],
        edgecolor="black",
    )
    ax.errorbar(
        centers,
        heights,
        yerr=yerr,
        fmt="none",
        ecolor="#2f2f2f",
        capsize=6,
        linewidth=1.2,
    )
    ax.set_xticks(centers)
    ax.set_xticklabels(stats_df["policy"], rotation=30, ha="right")
    ax.set_ylabel("Makespan (min)")
    ax.set_title("Median makespan with bootstrap CI")
fig.tight_layout()
fig.savefig(makespan_path, dpi=300, bbox_inches="tight")
plt.close(fig)
makespan_path

In [ ]:
site_path = ASSETS_DIR / f"{BACKEND}_site_selection_stacked_bar.png"
fig, ax = plt.subplots(figsize=(9, 5))
if site_rows.empty:
    ax.text(0.5, 0.5, "Site-level assignment data unavailable", ha="center", va="center")
    ax.axis("off")
else:
    totals = (
        site_rows.groupby(["policy", "site"], as_index=False)["assigned_jobs"].sum()
    )
    totals["share"] = totals.groupby("policy")["assigned_jobs"].transform(lambda s: s / s.sum())
    pivot = (
        totals.pivot_table(index="policy", columns="site", values="share", fill_value=0.0)
        .reindex(policy_order)
        .fillna(0.0)
    )
    bottom = np.zeros(len(pivot))
    sites = sorted(pivot.columns)
    for site in sites:
        values = pivot[site].to_numpy()
        ax.bar(
            np.arange(len(pivot)),
            values,
            bottom=bottom,
            label=site,
        )
        bottom += values
    ax.set_xticks(np.arange(len(pivot)))
    ax.set_xticklabels(pivot.index, rotation=30, ha="right")
    ax.set_ylabel("Share of assigned jobs")
    ax.set_ylim(0, 1)
    ax.legend(title="Site", bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.set_title("Site selection distribution")
fig.tight_layout()
fig.savefig(site_path, dpi=300, bbox_inches="tight")
plt.close(fig)
site_path

In [ ]:
sweep_ids = sorted(policy_rows["batch_id"].dropna().unique())
meta_path = write_run_metadata(BACKEND, sweep_ids, assets_dir=ASSETS_DIR)
meta_path